In [12]:
import os
from pathlib import Path  
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def connect_to_database(db_filename="database.sqlite"):
    """
    Zoekt dynamisch naar het databasebestand in de huidige werkmap of bovenliggende mappen
    en zet een veilige, foutbestendige SQLite-verbinding op.
    
    Args:
        db_filename (str): De exacte naam van het SQLite-bestand. Default is 'database.sqlite'.
        
    Returns:
        sqlite3.Connection: Een actieve databaseverbinding als het bestand bestaat.
        
    Raises:
        FileNotFoundError: Als het bestand nergens in de verwachte mappenstructuur wordt gevonden.
    """
    # Bepaal de huidige map waar het notebook draait 
    current_dir = Path.cwd()
    
    possible_paths = [
        current_dir / db_filename,
        current_dir / "notebook" / db_filename,
        current_dir.parent / "notebook" / db_filename,
        current_dir.parent / db_filename
    ]
    
    target_path = None
    for path in possible_paths:
        # Runtime fix: controleer of het bestand bestaat en niet leeg is
        if path.is_file() and path.stat().st_size > 0:
            target_path = path
            break
            
    # Harde fail-fast check om 'unsupported file format' errors door lege bestanden te voorkomen
    if not target_path:
        print("="*60)
        print("FOUT: Databasebestand kon niet automatisch worden gevonden!")
        print("Gecontroleerde locaties:")
        for path in possible_paths:
            print(f" - {path}")
        print("="*60)
        raise FileNotFoundError(f"Kon {db_filename} nergens vinden. Controleer de bestandsnaam.")
        
    # Bestand wel gevonden: log statistieken en maak de verbinding
    file_size_mb = target_path.stat().st_size / (1024 * 1024)
    print(f"✓ Database succesvol gelokaliseerd!")
    print(f"  Pad: {target_path}")
    print(f"  Grootte: {file_size_mb:.2f} MB")
    
    # Open de database in Read-Only modus (?mode=ro). 
    # Dit voorkomt dat SQLite stilletjes een leeg bestand aanmaakt bij een typefout.
    db_uri = f"file:{target_path}?mode=ro"
    connection = sqlite3.connect(db_uri, uri=True)
    print("✓ Succesvol verbonden met de database!")
    
    return connection

conn = connect_to_database()


✓ Database succesvol gelokaliseerd!
  Pad: c:\Users\sasha\Documents\GitHub\Datalab_semester2_Groep1\notebook\database.sqlite
  Grootte: 298.59 MB
✓ Succesvol verbonden met de database!


Set up

In [13]:
class BookmakerAnalyzer:
    """
    Een klasse om bookmaker data te analyseren, odds om te rekenen naar kansen
    en de ingebouwde winstmarges (overrounds) te berekenen.
    """
    
    def __init__(self, connection):
        """
        Initialiseert de analyzer met een actieve SQLite databaseverbinding.
        
        Args:
            connection (sqlite3.Connection): Actieve databaseverbinding.
        """
        self.conn = connection

    def load_match_odds(self, bookmaker_prefix="GB"):
        """
        Haalt de wedstrijdgegevens en de odds van de specifieke bookmaker op uit de Match-tabel.
        In de database worden de odds voor GBA vaak aangeduid met 'GBH' (Home), 'GBD' (Draw), 'GBA' (Away).
        
        Args:
            bookmaker_prefix (str): De prefix van de bookmaker in de database (bijv. 'GB' voor GBA).
            
        Returns:
            pd.DataFrame: DataFrame met de opgevraagde wedstrijd- en odds-data.
        """
        # We selecteren de kolommen voor Home (H), Draw (D) en Away (A) odds
        query = f"""
        SELECT 
            match_api_id, 
            home_team_goal,
            away_team_goal,
            home_team_api_id, 
            away_team_api_id, 
            {bookmaker_prefix}H AS odds_home, 
            {bookmaker_prefix}D AS odds_draw, 
            {bookmaker_prefix}A AS odds_away 
        FROM Match
        WHERE {bookmaker_prefix}H IS NOT NULL 
          AND {bookmaker_prefix}D IS NOT NULL 
          AND {bookmaker_prefix}A IS NOT NULL;
        """
        df = pd.read_sql_query(query, self.conn)
        return df

    @staticmethod
    def convert_odds_to_probabilities(df):
        """
        Rekent de decimale odds om naar impliciete kansen (probabilities).
        Formule: Kans = 1 / Odds
        
        Args:
            df (pd.DataFrame): DataFrame met de odds kolommen.
            
        Returns:
            pd.DataFrame: DataFrame uitgebreid met de berekende kansen en de totale marge.
        """
        df_prob = df.copy()
        
        # Bereken de impliciete kansen
        df_prob['prob_home'] = 1 / df_prob['odds_home']
        df_prob['prob_draw'] = 1 / df_prob['odds_draw']
        df_prob['prob_away'] = 1 / df_prob['odds_away']
        
        # Bereken de totale som van de kansen per wedstrijd
        df_prob['total_probability'] = df_prob['prob_home'] + df_prob['prob_draw'] + df_prob['prob_away']
        
        return df_prob

# 1. Initialiseer de klasse met de bestaande databaseverbinding
analyzer = BookmakerAnalyzer(conn)

# 2. Laad de data voor bookmaker GBA (gebruikt de 'GB' kolommen in de database)
df_gba_odds = analyzer.load_match_odds(bookmaker_prefix="GB")

# 3. Bereken de kansen
df_gba_analysis = analyzer.convert_odds_to_probabilities(df_gba_odds)

# Toon de eerste 5 rijen van het resultaat
print(f"Aantal succesvol opgehaalde wedstrijden voor GBA: {len(df_gba_analysis)}")
df_gba_analysis[['odds_home', 'odds_draw', 'odds_away', 'prob_home', 'prob_draw', 'prob_away', 'total_probability']].head()


Aantal succesvol opgehaalde wedstrijden voor GBA: 14162


,odds_home,odds_draw,odds_away,prob_home,prob_draw,prob_away,total_probability
0,1.78,3.25,4.00,0.561798,0.307692,0.250000,1.119490
1,1.85,3.25,3.75,0.540541,0.307692,0.266667,1.114900
2,2.50,3.20,2.50,0.400000,0.312500,0.400000,1.112500
3,1.50,3.75,5.50,0.666667,0.266667,0.181818,1.115152
4,4.50,3.50,1.65,0.222222,0.285714,0.606061,1.113997


Wat direct opvalt is dat de opgetelde kansen (total_probability) bij elke wedstrijd ongeveer uitkomen rond de 111% tot 112% in plaats van de logische 100%. Dit verschil van 11% tot 12% vertegenwoordigt de ingebouwde winstmarge van bookmaker GBA. Dit laat zien dat de ruwe kansen kunstmatig zijn 'opgeblazen' zodat de bookmaker altijd winst maakt, wat betekent dat we deze kansen eerst moeten normaliseren naar 100% voordat we ze eerlijk kunnen vergelijken met ons eigen voorspelmodel.

In [14]:
# Normaliseer de impliciete kansen van bookmaker GBA naar exact 100% (som = 1.0)
df_gba_analysis['prob_home_norm'] = df_gba_analysis['prob_home'] / df_gba_analysis['total_probability']
df_gba_analysis['prob_draw_norm'] = df_gba_analysis['prob_draw'] / df_gba_analysis['total_probability']
df_gba_analysis['prob_away_norm'] = df_gba_analysis['prob_away'] / df_gba_analysis['total_probability']

# Bepaal het voorspelde resultaat van de bookmaker (de uitslag met de hoogste kans)
def get_prediction(row):
    probs = {
        'win': row['prob_home_norm'],
        'draw': row['prob_draw_norm'],
        'defeat': row['prob_away_norm']
    }
    return max(probs, key=probs.get)

df_gba_analysis['bookmaker_prediction'] = df_gba_analysis.apply(get_prediction, axis=1)

# Toon de eerste 5 rijen met de genormaliseerde kolommen en de voorspelling ter controle
df_gba_analysis[['odds_home', 'odds_draw', 'odds_away', 'prob_home_norm', 'prob_draw_norm', 'prob_away_norm', 'bookmaker_prediction']].head()


,odds_home,odds_draw,odds_away,prob_home_norm,prob_draw_norm,prob_away_norm,bookmaker_prediction
0,1.78,3.25,4.00,0.501834,0.274850,0.223316,win
1,1.85,3.25,3.75,0.484833,0.275982,0.239184,win
2,2.50,3.20,2.50,0.359551,0.280899,0.359551,win
3,1.50,3.75,5.50,0.597826,0.239130,0.163043,win
4,4.50,3.50,1.65,0.199482,0.256477,0.544041,defeat


### 1.2 Voorspeld resultaat van de bookmaker bepalen

Om het voorspelde resultaat van de bookmaker te bepalen, hebben we de berekende kansen genormaliseerd zodat ze per wedstrijd exact optellen tot 100%. Hiermee wordt de winstmarge van de bookmaker opgeheven. Vervolgens selecteren we de uitkomst met de hoogste kans als de voorspelling van de bookmaker.

1.3

In [18]:
# 1. Laad de data opnieuw in (nu mét de doelpunten-kolommen)
df_gba_odds = analyzer.load_match_odds(bookmaker_prefix="GB")
df_gba_analysis = analyzer.convert_odds_to_probabilities(df_gba_odds)

# 2. Normaliseer de kansen
df_gba_analysis['prob_home_norm'] = df_gba_analysis['prob_home'] / df_gba_analysis['total_probability']
df_gba_analysis['prob_draw_norm'] = df_gba_analysis['prob_draw'] / df_gba_analysis['total_probability']
df_gba_analysis['prob_away_norm'] = df_gba_analysis['prob_away'] / df_gba_analysis['total_probability']

# 3. Bepaal de voorspelling van de bookmaker
df_gba_analysis['bookmaker_prediction'] = df_gba_analysis.apply(get_prediction, axis=1)

# 4. Bepaal het werkelijke resultaat (dit werkt nu wel!)
def get_actual_result(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 'win'
    elif row['home_team_goal'] < row['away_team_goal']:
        return 'defeat'
    else:
        return 'draw'

df_gba_analysis['actual_result'] = df_gba_analysis.apply(get_actual_result, axis=1)

# 5. Bereken de nauwkeurigheid (accuracy) van de bookmaker
bm_accuracy = (df_gba_analysis['bookmaker_prediction'] == df_gba_analysis['actual_result']).mean()
print(f"De bookmaker had de uitslag in {bm_accuracy:.1%} van de gevallen correct voorspeld.")

De bookmaker had de uitslag in 53.3% van de gevallen correct voorspeld.
